# Phase 2 — Measure
## 02 — Product Master & Canonical Key Mapping

### Objective
Build a governed Product Master that reconciles product identifiers across:

- Master Sales Data: `Stock Code`
- Stock on Hand: `Model`
- Master SOA: `Model`

### Goals
- Standardize product identifiers.
- Measure exact cross-source match coverage.
- Identify unmatched or ambiguous products.
- Detect duplicate identifiers.
- Create one canonical product key.
- Produce the governed `dim_product` table.

### Important Rule
No fuzzy matching or manual correction will be applied until exact normalized-key matching has been fully assessed.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_path = Path.cwd()

if current_path.name == "Phase_2_Measure":
    PROJECT_ROOT = current_path.parent.parent
elif current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
Raw data dir : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\raw
Processed dir: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed


In [2]:
## Locate the three source files
sales_files = list(RAW_DIR.glob("*Master*Sales*.xlsx"))
stock_files = list(RAW_DIR.glob("*Stock*Hand*.xlsx"))
soa_files = list(RAW_DIR.glob("*SOA*.xlsx"))

print("Sales files found :", len(sales_files))
for f in sales_files:
    print(" -", f.name)

print("\nStock files found :", len(stock_files))
for f in stock_files:
    print(" -", f.name)

print("\nSOA files found :", len(soa_files))
for f in soa_files:
    print(" -", f.name)

Sales files found : 1
 - Master_Sales_data.xlsx

Stock files found : 1
 - Stock on Hand Report.xlsx

SOA files found : 1
 - Master-SOA.xlsx


In [3]:
if len(sales_files) != 1:
    raise ValueError(
        f"Expected exactly 1 Master Sales workbook, found {len(sales_files)}"
    )

if len(stock_files) != 1:
    raise ValueError(
        f"Expected exactly 1 Stock-on-Hand workbook, found {len(stock_files)}"
    )

if len(soa_files) != 1:
    raise ValueError(
        f"Expected exactly 1 SOA workbook, found {len(soa_files)}"
    )

SALES_FILE = sales_files[0]
STOCK_FILE = stock_files[0]
SOA_FILE = soa_files[0]

print("Sales :", SALES_FILE.name)
print("Stock :", STOCK_FILE.name)
print("SOA   :", SOA_FILE.name)

Sales : Master_Sales_data.xlsx
Stock : Stock on Hand Report.xlsx
SOA   : Master-SOA.xlsx


In [4]:
## Inspect workbook sheets
sales_excel = pd.ExcelFile(SALES_FILE)
stock_excel = pd.ExcelFile(STOCK_FILE)
soa_excel = pd.ExcelFile(SOA_FILE)

print("Sales sheets :", sales_excel.sheet_names)
print("Stock sheets :", stock_excel.sheet_names)
print("SOA sheets   :", soa_excel.sheet_names)

Sales sheets : ['Master_data', 'Nov_file', 'Dec_file', 'Jan_file']
Stock sheets : ['Sheet1']
SOA sheets   : ['Master_SOA']


In [5]:
## Load the relevant tables
sales_master = pd.read_excel(
    SALES_FILE,
    sheet_name="Master_data"
)

stock_raw = pd.read_excel(
    STOCK_FILE,
    sheet_name=stock_excel.sheet_names[0]
)

soa_raw = pd.read_excel(
    SOA_FILE,
    sheet_name=soa_excel.sheet_names[0]
)

print("Sales Master shape :", sales_master.shape)
print("Stock shape        :", stock_raw.shape)
print("SOA shape          :", soa_raw.shape)

Sales Master shape : (966, 11)
Stock shape        : (383, 11)
SOA shape          : (412, 5)


In [6]:
## Confirm identifier columns
print("Sales identifier columns:")
print(
    sales_master[
        ["Stock Code", "Description", "Category"]
    ].head()
)

print("\nStock identifier columns:")
print(
    stock_raw[
        ["Model", "Description", "Category"]
    ].head()
)

print("\nSOA identifier columns:")
print(
    soa_raw[
        ["Model", "Description"]
    ].head()
)

Sales identifier columns:
   Stock Code                   Description     Category
0  TLS169BOXE  Boxed Uni Floor Tool 30-38MM  ACCESSORIES
1     112.204            TV Arial Lead 4.0m  ACCESSORIES
2     DLSC500             Delonghi Descaler  ACCESSORIES
3        AF01       VACUUM FRESHENERS AF101  ACCESSORIES
4  SES007NEU0     Sage Descaler (pack of 4)  ACCESSORIES

Stock identifier columns:
         Model                                        Description  \
0    ERACIXA60              Elica 52cm Canopy Hood for 60cm units   
1      P2152CH                        Powerpoint 52cm Canopy Hood   
2  INTEGRATA60  Elica 5414601 60cm Integrata Integrated Hood Grey   
3    DGE5861HM                        AEG 80cm Canopy Cooker Hood   
4    P2110XBSS                Powerpoint 60cm SS Traditional Hood   

       Category  
0  COOKER HOODS  
1  COOKER HOODS  
2  COOKER HOODS  
3  COOKER HOODS  
4  COOKER HOODS  

SOA identifier columns:
            Model                   Description
0  DW60A8

### Step 1 — Identifier Quality Profiling

In [7]:
## Profile key counts
key_profile = pd.DataFrame({
    "Source": [
        "Sales_Master",
        "Stock_On_Hand",
        "Master_SOA"
    ],
    "Rows": [
        len(sales_master),
        len(stock_raw),
        len(soa_raw)
    ],
    "Unique_Product_Keys": [
        sales_master["Stock Code"].nunique(dropna=True),
        stock_raw["Model"].nunique(dropna=True),
        soa_raw["Model"].nunique(dropna=True)
    ],
    "Missing_Product_Keys": [
        sales_master["Stock Code"].isna().sum(),
        stock_raw["Model"].isna().sum(),
        soa_raw["Model"].isna().sum()
    ],
    "Duplicate_Product_Keys": [
        sales_master["Stock Code"].duplicated().sum(),
        stock_raw["Model"].duplicated().sum(),
        soa_raw["Model"].duplicated().sum()
    ]
})

display(key_profile)

,Source,Rows,Unique_Product_Keys,Missing_Product_Keys,Duplicate_Product_Keys
0,Sales_Master,966,768,0,198
1,Stock_On_Hand,383,383,0,0
2,Master_SOA,412,412,0,0


In [8]:
## Inspect duplicated keys by source
sales_duplicate_keys = (
    sales_master[
        sales_master["Stock Code"].duplicated(
            keep=False
        )
    ]
    .sort_values("Stock Code")
)

stock_duplicate_keys = (
    stock_raw[
        stock_raw["Model"].duplicated(
            keep=False
        )
    ]
    .sort_values("Model")
)

soa_duplicate_keys = (
    soa_raw[
        soa_raw["Model"].duplicated(
            keep=False
        )
    ]
    .sort_values("Model")
)

print("Sales rows with duplicated Stock Code:",
      len(sales_duplicate_keys))

print("Stock rows with duplicated Model:",
      len(stock_duplicate_keys))

print("SOA rows with duplicated Model:",
      len(soa_duplicate_keys))

Sales rows with duplicated Stock Code: 365
Stock rows with duplicated Model: 0
SOA rows with duplicated Model: 0


In [9]:
## Check whitespace / formatting anomalies
def identifier_quality(series):
    s = series.dropna().astype(str)

    return {
        "records": len(s),
        "leading_trailing_spaces": (
            s != s.str.strip()
        ).sum(),
        "lowercase_present": (
            s != s.str.upper()
        ).sum(),
        "blank_after_strip": (
            s.str.strip() == ""
        ).sum()
    }


identifier_quality_df = pd.DataFrame([
    {
        "Source": "Sales",
        **identifier_quality(
            sales_master["Stock Code"]
        )
    },
    {
        "Source": "Stock",
        **identifier_quality(
            stock_raw["Model"]
        )
    },
    {
        "Source": "SOA",
        **identifier_quality(
            soa_raw["Model"]
        )
    }
])

display(identifier_quality_df)

,Source,records,leading_trailing_spaces,lowercase_present,blank_after_strip
0,Sales,966,0,137,0
1,Stock,383,0,0,0
2,SOA,412,0,0,0


In [10]:
## Create normalized keys
def normalize_product_key(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
    )


sales_master["Product_Key_Normalized"] = normalize_product_key(
    sales_master["Stock Code"]
)

stock_raw["Product_Key_Normalized"] = normalize_product_key(
    stock_raw["Model"]
)

soa_raw["Product_Key_Normalized"] = normalize_product_key(
    soa_raw["Model"]
)

print("Normalized product keys created.")

Normalized product keys created.


In [11]:
## Check whether normalization creates collisions
def normalization_collision_check(df, raw_col):
    check = (
        df[[raw_col, "Product_Key_Normalized"]]
        .drop_duplicates()
        .groupby("Product_Key_Normalized")[raw_col]
        .nunique()
    )

    return check[check > 1]


sales_collisions = normalization_collision_check(
    sales_master,
    "Stock Code"
)

stock_collisions = normalization_collision_check(
    stock_raw,
    "Model"
)

soa_collisions = normalization_collision_check(
    soa_raw,
    "Model"
)

print("Sales normalization collisions:", len(sales_collisions))
print("Stock normalization collisions :", len(stock_collisions))
print("SOA normalization collisions   :", len(soa_collisions))

Sales normalization collisions: 1
Stock normalization collisions : 0
SOA normalization collisions   : 0


In [12]:
## Build unique key sets
sales_keys = set(
    sales_master["Product_Key_Normalized"].dropna().unique()
)

stock_keys = set(
    stock_raw["Product_Key_Normalized"].dropna().unique()
)

soa_keys = set(
    soa_raw["Product_Key_Normalized"].dropna().unique()
)

print("Unique Sales keys :", len(sales_keys))
print("Unique Stock keys :", len(stock_keys))
print("Unique SOA keys   :", len(soa_keys))

Unique Sales keys : 767
Unique Stock keys : 383
Unique SOA keys   : 412


In [13]:
## Exact Cross-Source Matching 
sales_stock_match = sales_keys & stock_keys
sales_soa_match = sales_keys & soa_keys
stock_soa_match = stock_keys & soa_keys
all_three_match = sales_keys & stock_keys & soa_keys

print("Sales ↔ Stock exact matches :", len(sales_stock_match))
print("Sales ↔ SOA exact matches   :", len(sales_soa_match))
print("Stock ↔ SOA exact matches   :", len(stock_soa_match))
print("Present in all 3 sources    :", len(all_three_match))

Sales ↔ Stock exact matches : 56
Sales ↔ SOA exact matches   : 62
Stock ↔ SOA exact matches   : 8
Present in all 3 sources    : 0


In [14]:
match_coverage = pd.DataFrame({
    "Comparison": [
        "Sales found in Stock",
        "Stock found in Sales",
        "Sales found in SOA",
        "SOA found in Sales",
        "Stock found in SOA",
        "SOA found in Stock"
    ],
    "Matched": [
        len(sales_keys & stock_keys),
        len(stock_keys & sales_keys),
        len(sales_keys & soa_keys),
        len(soa_keys & sales_keys),
        len(stock_keys & soa_keys),
        len(soa_keys & stock_keys)
    ],
    "Source_Total": [
        len(sales_keys),
        len(stock_keys),
        len(sales_keys),
        len(soa_keys),
        len(stock_keys),
        len(soa_keys)
    ]
})

match_coverage["Coverage_%"] = (
    match_coverage["Matched"]
    / match_coverage["Source_Total"]
    * 100
).round(2)

display(match_coverage)

,Comparison,Matched,Source_Total,Coverage_%
0,Sales found in Stock,56,767,7.30
1,Stock found in Sales,56,383,14.62
2,Sales found in SOA,62,767,8.08
3,SOA found in Sales,62,412,15.05
4,Stock found in SOA,8,383,2.09
5,SOA found in Stock,8,412,1.94


In [15]:
## Source membership matrix
all_product_keys = sorted(
    sales_keys | stock_keys | soa_keys
)

product_membership = pd.DataFrame({
    "Product_Key": all_product_keys
})

product_membership["In_Sales"] = (
    product_membership["Product_Key"].isin(sales_keys)
)

product_membership["In_Stock"] = (
    product_membership["Product_Key"].isin(stock_keys)
)

product_membership["In_SOA"] = (
    product_membership["Product_Key"].isin(soa_keys)
)

product_membership["Source_Count"] = (
    product_membership[
        ["In_Sales", "In_Stock", "In_SOA"]
    ].sum(axis=1)
)

print(
    "Total distinct products across ecosystem:",
    len(product_membership)
)

display(
    product_membership["Source_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Number_of_Sources")
    .to_frame("Products")
)

Total distinct products across ecosystem: 1436


,Products
Number_of_Sources,
1,1310
2,126


In [16]:
membership_patterns = (
    product_membership
    .groupby(
        ["In_Sales", "In_Stock", "In_SOA"]
    )
    .size()
    .reset_index(name="Product_Count")
    .sort_values(
        "Product_Count",
        ascending=False
    )
)

display(membership_patterns)

,In_Sales,In_Stock,In_SOA,Product_Count
3,True,False,False,649
0,False,False,True,342
1,False,True,False,319
4,True,False,True,62
5,True,True,False,56
2,False,True,True,8


### Step 2 — Investigate the Normalization Collision

In [22]:
print("Sales normalization collision(s):")

display(
    sales_master[
        sales_master["Product_Key_Normalized"].isin(
            sales_collisions.index
        )
    ][
        [
            "Stock Code",
            "Product_Key_Normalized",
            "Description",
            "Category"
        ]
    ]
    .sort_values(
        ["Product_Key_Normalized", "Stock Code"]
    )
)

Sales normalization collision(s):


,Stock Code,Product_Key_Normalized,Description,Category
61,DELIVERY,DELIVERY,Web Delivery Charge,DELIVERY CHARGE
440,DELIVERY,DELIVERY,Web Delivery Charge,DELIVERY CHARGE
749,DELIVERY,DELIVERY,Web Delivery Charge,DELIVERY CHARGE
60,Delivery,DELIVERY,WEB DELIVERY CHARGE,DELIVERY CHARGE
439,Delivery,DELIVERY,WEB DELIVERY CHARGE,DELIVERY CHARGE
748,Delivery,DELIVERY,WEB DELIVERY CHARGE,DELIVERY CHARGE


####  Step 2.1 — Investigate Why Exact Matching Is So Low

In [17]:
## Build unique source product tables

sales_products = (
    sales_master[
        [
            "Stock Code",
            "Product_Key_Normalized",
            "Description",
            "Category"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Stock Code": "Sales_Stock_Code",
            "Description": "Sales_Description",
            "Category": "Sales_Category"
        }
    )
)

stock_products = (
    stock_raw[
        [
            "Model",
            "Product_Key_Normalized",
            "Description",
            "Category"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Model": "Stock_Model",
            "Description": "Stock_Description",
            "Category": "Stock_Category"
        }
    )
)

soa_products = (
    soa_raw[
        [
            "Model",
            "Product_Key_Normalized",
            "Description"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Model": "SOA_Model",
            "Description": "SOA_Description"
        }
    )
)

print("Sales products :", len(sales_products))
print("Stock products :", len(stock_products))
print("SOA products   :", len(soa_products))

Sales products : 767
Stock products : 383
SOA products   : 412


In [18]:
## Inspect exact Sales ↔ Stock matches
sales_stock_exact = sales_products.merge(
    stock_products,
    on="Product_Key_Normalized",
    how="inner"
)

print(
    "Exact Sales ↔ Stock products:",
    len(sales_stock_exact)
)

display(
    sales_stock_exact[
        [
            "Product_Key_Normalized",
            "Sales_Stock_Code",
            "Stock_Model",
            "Sales_Description",
            "Stock_Description",
            "Sales_Category",
            "Stock_Category"
        ]
    ].head(20)
)

Exact Sales ↔ Stock products: 56


,Product_Key_Normalized,Sales_Stock_Code,Stock_Model,Sales_Description,Stock_Description,Sales_Category,Stock_Category
0,DGE5861HM,DGE5861HM,DGE5861HM,AEG 80cm Canopy Cooker Hood,AEG 80cm Canopy Cooker Hood,COOKER HOODS,COOKER HOODS
1,LKR655200K,LKR655200K,LKR655200K,Electrolux 60cm Black Cooker,Electrolux 60cm Black Cooker,COOKERS,COOKERS
2,LKR555100B,LKR555100B,LKR555100B,Electrolux 55cm Black Cooker,Electrolux 55cm Black Cooker,COOKERS,COOKERS
3,KDFGE40TX,KDFGE40TX,KDFGE40TX,ELECTROLUX D/O ENAMEL,ELECTROLUX D/O ENAMEL LINERS STEEL,DOUBLE OVENS,DOUBLE OVENS
4,KGE49AICAG,KGE49AICAG,KGE49AICAG,*Bosch St/St 201x70 Fridge Freezer,*Bosch St/St 201x70 Fridge Freezer,FRIDGE FREEZERS,FRIDGE FREEZERS
5,KS36VVIEPG,KS36VVIEPG,KS36VVIEPG,Siemens Inox Fridge 186 x 60cm,Siemens Inox Fridge 186 x 60cm,FRIDGES,FRIDGES
6,PKE611CA3E,PKE611CA3E,PKE611CA3E,*Bosch 60cm Serie 2 Ceramic Hob,*Bosch 60cm Serie 2 Ceramic Hob with Knobs,HOBS,HOBS
7,SI2641D,SI2641D,SI2641D,Smeg 60cm 7.2kW Induction Hob,Smeg 60cm 7.2kW Induction Hob,HOBS,HOBS
8,T58FHW1L0,T58FHW1L0,T58FHW1L0,Neff N70 80cm Induction Hob,Neff N70 80cm Induction Hob,HOBS,HOBS
9,LIB60420C,LIB60420C,LIB60420C,Electrolux 60cm Induction Hob,Electrolux 60cm Induction Hob,HOBS,HOBS


In [19]:
## Inspect exact Sales ↔ SOA matches
sales_soa_exact = sales_products.merge(
    soa_products,
    on="Product_Key_Normalized",
    how="inner"
)

print(
    "Exact Sales ↔ SOA products:",
    len(sales_soa_exact)
)

display(
    sales_soa_exact[
        [
            "Product_Key_Normalized",
            "Sales_Stock_Code",
            "SOA_Model",
            "Sales_Description",
            "SOA_Description"
        ]
    ].head(20)
)

Exact Sales ↔ SOA products: 62


,Product_Key_Normalized,Sales_Stock_Code,SOA_Model,Sales_Description,SOA_Description
0,TAPOL530E,TAPOL530E,TAPOL530E,Tapo Smart WiFi Multicolur Screw,Tapo Smart WiFi Multicolur Screw
1,BES875UK,BES875UK,BES875UK,Sage St/St Barista Express Coffee,Sage St/St Barista Express Coffee
2,ES601UK,ES601UK,ES601UK,Ninja Luxe Café Premier series,Ninja Luxe Café Premier series
3,ES601UKBK,ES601UKBK,ES601UKBK,Ninja Luxe Café Premier Espresso,Ninja Luxe Café Premier Espresso
4,BN800UK,BN800UK,BN800UK,NINJA 3 IN 1 FOOD,NINJA 3 IN 1 FOOD
5,NC701UK,NC701UK,NC701UK,Ninja CREAMi Swirl,Ninja CREAMi Swirl
6,AF160UK,AF160UK,AF160UK,Ninja Air Fryer Max,Ninja Air Fryer Max
7,AF300UK,AF300UK,AF300UK,Ninja Dual Zone Air Fryer,Ninja Dual Zone Air Fryer
8,AF400UK,AF400UK,AF400UK,Ninja Foodi Max Dual Zon Fryer 9.5L,Ninja Foodi Max Dual Zon Fryer
9,AF500UK,AF500UK,AF500UK,Ninja Foodi FlexDrawer 10.4L,Ninja Foodi FlexDrawer 10.4L


In [21]:
stock_soa_exact = stock_products.merge(
    soa_products,
    on="Product_Key_Normalized",
    how="inner"
)

print("Exact Stock ↔ SOA products:", len(stock_soa_exact))

display(
    stock_soa_exact[
        [
            "Product_Key_Normalized",
            "Stock_Model",
            "SOA_Model",
            "Stock_Description",
            "SOA_Description"
        ]
    ]
)

Exact Stock ↔ SOA products: 8


,Product_Key_Normalized,Stock_Model,SOA_Model,Stock_Description,SOA_Description
0,MC28H5013AS/EU,MC28H5013AS/EU,MC28H5013AS/EU,Samsung Silver 28L Microwave,Samsung Silver 28L Microwave
1,DV90DB8845GBU1,DV90DB8845GBU1,DV90DB8845GBU1,Samsung Series 8 9kg Heat Pump Dryer,Samsung Series 8 9kg Heat Pump
2,DV90DG52A0AEEU,DV90DG52A0AEEU,DV90DG52A0AEEU,Samsung Series 5 Tumble Dryer,Samsung Series 5 Tumble Dryer
3,DV90DG6845LBU1,DV90DG6845LBU1,DV90DG6845LBU1,Samsung Series 6 9kg Heat Pump Dryer,Samsung Series 6 9kg Heat Pump
4,DV90DG6845LEU1,DV90DG6845LEU1,DV90DG6845LEU1,Sumsung White 9kg Heat Pump Dryer,Sumsung White 9kg Heat Pump
5,RS70F64KEFEU,RS70F64KEFEU,RS70F64KEFEU,Samsung Black St/St USA FF,Samsung Black St/St USA FF
6,RS70F64KETEU,RS70F64KETEU,RS70F64KETEU,Samsung Silver USA Fridge Freezer,Samsung USA Fridge Freezer
7,RS90F66BEFEU,RS90F66BEFEU,RS90F66BEFEU,Samsung Black Family Hob USA FF,Samsung Black Family Hob USA


### Step 3 — Build the Canonical Product Master

In [23]:
sales_dim_source = (
    sales_master[
        [
            "Product_Key_Normalized",
            "Stock Code",
            "Description",
            "Category"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Stock Code": "Sales_Stock_Code",
            "Description": "Sales_Description",
            "Category": "Sales_Category"
        }
    )
)

stock_dim_source = (
    stock_raw[
        [
            "Product_Key_Normalized",
            "Model",
            "Description",
            "Category"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Model": "Stock_Model",
            "Description": "Stock_Description",
            "Category": "Stock_Category"
        }
    )
)

soa_dim_source = (
    soa_raw[
        [
            "Product_Key_Normalized",
            "Model",
            "Description"
        ]
    ]
    .drop_duplicates(
        subset=["Product_Key_Normalized"]
    )
    .rename(
        columns={
            "Model": "SOA_Model",
            "Description": "SOA_Description"
        }
    )
)

In [24]:
## Build complete Product Master universe
dim_product = pd.DataFrame({
    "Product_Key": sorted(
        sales_keys | stock_keys | soa_keys
    )
})

dim_product = (
    dim_product
    .merge(
        sales_dim_source,
        left_on="Product_Key",
        right_on="Product_Key_Normalized",
        how="left"
    )
    .drop(columns="Product_Key_Normalized")
)

dim_product = (
    dim_product
    .merge(
        stock_dim_source,
        left_on="Product_Key",
        right_on="Product_Key_Normalized",
        how="left"
    )
    .drop(columns="Product_Key_Normalized")
)

dim_product = (
    dim_product
    .merge(
        soa_dim_source,
        left_on="Product_Key",
        right_on="Product_Key_Normalized",
        how="left"
    )
    .drop(columns="Product_Key_Normalized")
)

print("Product Master rows:", len(dim_product))

display(dim_product.head())

Product Master rows: 1436


,Product_Key,Sales_Stock_Code,Sales_Description,Sales_Category,Stock_Model,Stock_Description,Stock_Category,SOA_Model,SOA_Description
0,010-02384-10,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,NaN,NaN,NaN,NaN,NaN
1,010-02784-00,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,NaN,NaN,NaN,NaN,NaN
2,010-02784-01,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,NaN,NaN,NaN,NaN,NaN
3,010-02839-00,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,NaN,NaN,NaN,NaN,NaN
4,01950,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,NaN,NaN,NaN,NaN,NaN


In [25]:
## Add source-presence flags
dim_product["In_Sales"] = (
    dim_product["Sales_Stock_Code"].notna()
)

dim_product["In_Stock"] = (
    dim_product["Stock_Model"].notna()
)

dim_product["In_SOA"] = (
    dim_product["SOA_Model"].notna()
)

dim_product["Source_Count"] = (
    dim_product[
        ["In_Sales", "In_Stock", "In_SOA"]
    ].sum(axis=1)
)

In [26]:
## Choose governed description and category

dim_product["Product_Description"] = (
    dim_product["Stock_Description"]
    .combine_first(
        dim_product["Sales_Description"]
    )
    .combine_first(
        dim_product["SOA_Description"]
    )
)

dim_product["Product_Category"] = (
    dim_product["Stock_Category"]
    .combine_first(
        dim_product["Sales_Category"]
    )
)

In [27]:
## Add identity/match status
conditions = [
    dim_product["Source_Count"] == 3,

    (
        dim_product["In_Sales"]
        & dim_product["In_Stock"]
    ),

    (
        dim_product["In_Sales"]
        & dim_product["In_SOA"]
    ),

    (
        dim_product["In_Stock"]
        & dim_product["In_SOA"]
    ),

    dim_product["In_Sales"],
    dim_product["In_Stock"],
    dim_product["In_SOA"]
]

choices = [
    "SALES_STOCK_SOA",
    "SALES_STOCK",
    "SALES_SOA",
    "STOCK_SOA",
    "SALES_ONLY",
    "STOCK_ONLY",
    "SOA_ONLY"
]

dim_product["Source_Status"] = np.select(
    conditions,
    choices,
    default="UNKNOWN"
)

display(
    dim_product["Source_Status"]
    .value_counts()
    .to_frame("Product_Count")
)

,Product_Count
Source_Status,
SALES_ONLY,649
SOA_ONLY,342
STOCK_ONLY,319
SALES_SOA,62
SALES_STOCK,56
STOCK_SOA,8


In [28]:
## Add surrogate Product ID
dim_product = (
    dim_product
    .sort_values("Product_Key")
    .reset_index(drop=True)
)

dim_product.insert(
    0,
    "Product_ID",
    range(1, len(dim_product) + 1)
)

display(
    dim_product[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Source_Status"
        ]
    ].head(10)
)

,Product_ID,Product_Key,Product_Description,Product_Category,Source_Status
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,SALES_ONLY
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,SALES_ONLY
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,SALES_ONLY
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,SALES_ONLY
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,SALES_ONLY
5,6,10009310,Miele GGRP Gourmet Griddle Plate,WHITES ACCESSORIES,SALES_ONLY
6,7,10107860,Miele SF-AP 50 Air Clean Plus Filter,VACUUM ACCESSORIES,SALES_ONLY
7,8,10234470,Miele Nature Flacon,WHITES ACCESSORIES,SALES_ONLY
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES,SALES_ONLY
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES,SALES_ONLY


In [29]:
## Product Master integrity checks

print("Rows:", len(dim_product))

print(
    "Unique Product_ID:",
    dim_product["Product_ID"].nunique()
)

print(
    "Unique Product_Key:",
    dim_product["Product_Key"].nunique()
)

print(
    "Duplicate Product_Key:",
    dim_product["Product_Key"].duplicated().sum()
)

print(
    "Missing Product_Key:",
    dim_product["Product_Key"].isna().sum()
)

print(
    "Missing Description:",
    dim_product["Product_Description"].isna().sum()
)

print(
    "Missing Category:",
    dim_product["Product_Category"].isna().sum()
)

Rows: 1436
Unique Product_ID: 1436
Unique Product_Key: 1436
Duplicate Product_Key: 0
Missing Product_Key: 0
Missing Description: 0
Missing Category: 342


### Step 4 — Product vs Non-Product Classification

In [37]:
dim_product["Record_Type"] = "PRODUCT"

service_charge_keys = [
    "DELIVERY",
    "DELIVERY-CHLOCAL"
]

dim_product.loc[
    dim_product["Product_Key"].isin(service_charge_keys),
    "Record_Type"
] = "SERVICE_CHARGE"

display(
    dim_product[
        dim_product["Record_Type"] != "PRODUCT"
    ][
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Record_Type"
        ]
    ]
)

,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type
368,369,DELIVERY,WEB DELIVERY CHARGE,DELIVERY CHARGE,SERVICE_CHARGE
369,370,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,SERVICE_CHARGE


In [38]:
## Explicitly classify DELIVERY
dim_product["Record_Type"] = "PRODUCT"

dim_product.loc[
    dim_product["Product_Key"] == "DELIVERY",
    "Record_Type"
] = "SERVICE_CHARGE"

display(
    dim_product[
        dim_product["Record_Type"] != "PRODUCT"
    ][
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Record_Type"
        ]
    ]
)

,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type
368,369,DELIVERY,WEB DELIVERY CHARGE,DELIVERY CHARGE,SERVICE_CHARGE


### Step 5 — Referential Integrity Test

In [39]:
product_key_map = (
    dim_product[
        [
            "Product_ID",
            "Product_Key"
        ]
    ]
    .copy()
)

print("Mapping rows       :", len(product_key_map))
print(
    "Unique Product_Key :",
    product_key_map["Product_Key"].nunique()
)

print(
    "Unique Product_ID  :",
    product_key_map["Product_ID"].nunique()
)

Mapping rows       : 1436
Unique Product_Key : 1436
Unique Product_ID  : 1436


In [40]:
## Test Sales mapping
sales_mapping_test = sales_master.merge(
    product_key_map,
    left_on="Product_Key_Normalized",
    right_on="Product_Key",
    how="left",
    validate="many_to_one"
)

print("Sales source rows :", len(sales_master))
print("Sales mapped rows :", len(sales_mapping_test))

print(
    "Unmapped Sales rows:",
    sales_mapping_test["Product_ID"].isna().sum()
)

Sales source rows : 966
Sales mapped rows : 966
Unmapped Sales rows: 0


In [41]:
## Test Stock mapping
stock_mapping_test = stock_raw.merge(
    product_key_map,
    left_on="Product_Key_Normalized",
    right_on="Product_Key",
    how="left",
    validate="many_to_one"
)

print("Stock source rows :", len(stock_raw))
print("Stock mapped rows :", len(stock_mapping_test))

print(
    "Unmapped Stock rows:",
    stock_mapping_test["Product_ID"].isna().sum()
)

Stock source rows : 383
Stock mapped rows : 383
Unmapped Stock rows: 0


In [42]:
## ## Test SOA mapping
soa_mapping_test = soa_raw.merge(
    product_key_map,
    left_on="Product_Key_Normalized",
    right_on="Product_Key",
    how="left",
    validate="many_to_one"
)

print("SOA source rows :", len(soa_raw))
print("SOA mapped rows :", len(soa_mapping_test))

print(
    "Unmapped SOA rows:",
    soa_mapping_test["Product_ID"].isna().sum()
)

SOA source rows : 412
SOA mapped rows : 412
Unmapped SOA rows: 0


In [48]:
product_master_gate = (
    len(dim_product) == 1436
    and dim_product["Product_ID"].is_unique
    and dim_product["Product_Key"].is_unique
    and dim_product["Product_Key"].notna().all()
    and dim_product["Product_Description"].notna().all()
    and sales_mapping_test["Product_ID"].notna().all()
    and stock_mapping_test["Product_ID"].notna().all()
    and soa_mapping_test["Product_ID"].notna().all()
)

print(
    "PRODUCT MASTER GATE:",
    "PASSED" if product_master_gate else "FAILED"
)

PRODUCT MASTER GATE: PASSED


### Step 6 — Final dim_product save

In [44]:
## Select governed columns
dim_product_final = dim_product[
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category",
        "Record_Type",
        "Source_Status",
        "Source_Count",
        "In_Sales",
        "In_Stock",
        "In_SOA",
        "Sales_Stock_Code",
        "Sales_Description",
        "Sales_Category",
        "Stock_Model",
        "Stock_Description",
        "Stock_Category",
        "SOA_Model",
        "SOA_Description"
    ]
].copy()

print("Final dim_product shape:", dim_product_final.shape)

display(dim_product_final.head())

Final dim_product shape: (1436, 18)


,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type,Source_Status,Source_Count,In_Sales,In_Stock,In_SOA,Sales_Stock_Code,Sales_Description,Sales_Category,Stock_Model,Stock_Description,Stock_Category,SOA_Model,SOA_Description
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,NaN,NaN,NaN,NaN,NaN
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,NaN,NaN,NaN,NaN,NaN
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,NaN,NaN,NaN,NaN,NaN
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,NaN,NaN,NaN,NaN,NaN
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,PRODUCT,SALES_ONLY,1,True,False,False,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,NaN,NaN,NaN,NaN,NaN


In [45]:
""" PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DIM_PRODUCT_FILE = (
    PROCESSED_DIR / "dim_product.csv"
)

dim_product_final.to_csv(
    DIM_PRODUCT_FILE,
    index=False
)

print("Saved:", DIM_PRODUCT_FILE) """

Saved: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\dim_product.csv


In [46]:
dim_product_check = pd.read_csv(
    DIM_PRODUCT_FILE
)

print("Saved rows:", len(dim_product_check))

print(
    "Unique Product_ID:",
    dim_product_check["Product_ID"].nunique()
)

print(
    "Unique Product_Key:",
    dim_product_check["Product_Key"].nunique()
)

print(
    "Duplicate Product_Key:",
    dim_product_check["Product_Key"].duplicated().sum()
)

print(
    "Missing Product_Key:",
    dim_product_check["Product_Key"].isna().sum()
)

print(
    "Missing Description:",
    dim_product_check["Product_Description"].isna().sum()
)

print("\nRecord types:")

display(
    dim_product_check["Record_Type"]
    .value_counts()
    .to_frame("Count")
)

Saved rows: 1436
Unique Product_ID: 1436
Unique Product_Key: 1436
Duplicate Product_Key: 0
Missing Product_Key: 0
Missing Description: 0

Record types:


,Count
Record_Type,
PRODUCT,1435
SERVICE_CHARGE,1
